<a href="https://colab.research.google.com/github/DariaChernykh4/DA-projects/blob/main/A_B_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Introduction**

This project presents a comprehensive A/B testing analysis report for an e-commerce platform, built using SQL, Python, and Tableau.

**Objective**

The goal of this analysis is to evaluate the impact of four A/B tests on key conversion metrics, assess statistical significance across multiple dimensions (device, continent, country, channel), and identify which variations drive meaningful improvements in user behavior — providing actionable recommendations for product and marketing teams.

**Dataset**

The dataset was extracted from Google BigQuery (DA dataset) and covers 87 days of user activity from November 1, 2020 to January 27, 2021. The data combines 4 distinct A/B tests, each with control (Group 1) and test (Group 2) variants.

**Methodology**

Statistical significance was determined using a two-proportion z-test with a 95% confidence level (α = 0.05). The analysis was performed dynamically through Python loops, allowing scalability to any number of metrics and grouping dimensions.

**Tools Used**

- SQL + Google BigQuery — data extraction, table joins, and initial aggregation
- Python (pandas, numpy, statsmodels) — data transformation, statistical significance calculation (z-test), and results export
- Tableau Public — interactive dashboard for visualizing test outcomes and significant metric changes

### [**View Interactive Dashboard (Tableau Public)**](https://public.tableau.com/shared/RKDNK4M3R?:display_count=n&:origin=viz_share_link)

## **2. Libraries Installation and Import**

In [ ]:
# Installing a library to work with Google BigQuery
!pip install google-cloud-bigquery pandas pyarrow

In [ ]:
import numpy as np
import pandas as pd

from google.colab import auth, files
from google.cloud import bigquery
from scipy import stats
import statsmodels.api as sm

## **3. Data Extraction (SQL + BigQuery)**

In [ ]:
# Authentication
auth.authenticate_user()

# Creating a BigQuery client
client = bigquery.Client(project="data-analytics-mate")

In [ ]:
# SQL Query
query = """
WITH
  session_info AS (
    SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
    FROM `data-analytics-mate.DA.ab_test` ab
    JOIN `DA.session` s
      ON ab.ga_session_id = s.ga_session_id
    JOIN `DA.session_params` sp
      ON s.ga_session_id = sp.ga_session_id
  ),
  session_with_orders AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT o.ga_session_id) AS session_with_orders
    FROM `DA.order` o
    JOIN session_info
      ON o.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  ),
  events AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name,
      COUNT(ep.ga_session_id) AS event_cnt
    FROM `DA.event_params` ep
    JOIN session_info
      ON ep.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name
  ),
  session AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
    FROM session_info
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  ),
  account AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
    FROM `DA.account_session` acs
    JOIN session_info
      ON acs.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  )


SELECT
  session_with_orders.date,
  session_with_orders.country,
  session_with_orders.device,
  session_with_orders.continent,
  session_with_orders.channel,
  session_with_orders.test,
  session_with_orders.test_group,
  'session_with_orders' AS event_name,
  session_with_orders.session_with_orders AS value
FROM session_with_orders
UNION ALL
SELECT
  events.date,
  events.country,
  events.device,
  events.continent,
  events.channel,
  events.test,
  events.test_group,
  event_name,
  event_cnt AS value
FROM events
UNION ALL
SELECT
  session.date,
  session.country,
  session.device,
  session.continent,
  session.channel,
  session.test,
  session.test_group,
  'session' AS event_name,
  session_cnt AS value
FROM session
UNION ALL
SELECT
  account.date,
  account.country,
  account.device,
  account.continent,
  account.channel,
  account.test,
  account.test_group,
  'new_accounts' AS event_name,
  new_account_cnt AS value
FROM account
"""

data = client.query(query).to_dataframe()
data.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-06,Slovakia,mobile,Europe,Paid Search,2,2,session_with_orders,1
1,2020-11-06,Slovakia,mobile,Europe,Paid Search,1,2,session_with_orders,1
2,2020-12-09,El Salvador,mobile,Americas,Direct,4,2,session_with_orders,1
3,2020-12-09,El Salvador,mobile,Americas,Direct,3,2,session_with_orders,1
4,2020-12-21,Slovakia,mobile,Europe,Organic Search,4,2,session_with_orders,1


## **4. Data Overview**

### **Dataset Size**

The dataset contains **800,996 rows** and **9 columns**.

In [ ]:
# Dataset Size

print(f"Total rows: {data.shape[0]:,}")
print(f"Total columns: {data.shape[1]}")

Total rows: 800,996
Total columns: 9


### **Data Types**

The dataset contains the following column types:

In [ ]:
print(data.dtypes)

date          dbdate
country       object
device        object
continent     object
channel       object
test           Int64
test_group     Int64
event_name    object
value          Int64
dtype: object


In [ ]:
numeric_columns = data.select_dtypes(include=["Int64","float64"]).columns
categorical_columns = data.select_dtypes(include=["object"]).columns
datetime_columns = data.select_dtypes(include=["dbdate"]).columns

print("\n Numeric columns:", len(numeric_columns))
print("Numeric columns names:", list(numeric_columns))
print("\n Categorical columns:", len(categorical_columns))
print("Categorical columns names:", list(categorical_columns))
print("\n Datetime columns:", len(datetime_columns))
print("Datetime columns names:", list(datetime_columns))


 Numeric columns: 3
Numeric columns names: ['test', 'test_group', 'value']

 Categorical columns: 5
Categorical columns names: ['country', 'device', 'continent', 'channel', 'event_name']

 Datetime columns: 1
Datetime columns names: ['date']


**Column Types**

**1. Datetime columns (1):**
- *date* - date when the session occurred and the A/B test event was recorded

**Note**: *date* data type is dbdate (BigQuery type) and should be transformed into standard pandas datetime.


**2. Numeric columns (3):**
- *test* - A/B test identifier
- *test_group* - group within the test
- *value* - count of events or sessions for the given row

**Note**: *test* and *test_group* are categorical by nature but stored as integers. They should be treated as categorical variables in analysis.

**3. Categorical columns (5):**
- *continent* - continent where the user is located
- *country* - user country determined by IP address
- *device* - type of device used (desktop, mobile, tablet)
- *channel* - aggregated traffic channel (organic, paid, direct, etc.)
- *event_name* - type of event recorded (session, add_payment_info, begin_checkout, new account, etc.)

In [ ]:
# Data transformation
data["date"] = pd.to_datetime(data["date"])

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date        800996 non-null  datetime64[ns]
 1   country     800996 non-null  object        
 2   device      800996 non-null  object        
 3   continent   800996 non-null  object        
 4   channel     800996 non-null  object        
 5   test        800996 non-null  Int64         
 6   test_group  800996 non-null  Int64         
 7   event_name  800996 non-null  object        
 8   value       800996 non-null  Int64         
dtypes: Int64(3), datetime64[ns](1), object(5)
memory usage: 57.3+ MB


### **Time Period**

In [ ]:
date_min = data["date"].min().date()
date_max = data["date"].max().date()
period = date_max - date_min

print("Period:", date_min, "to", date_max)
print("Duration", period.days, "days")

Period: 2020-11-01 to 2021-01-27
Duration 87 days


The dataset covers the period from **11/01/2020 to 01/27/2021**.

### **Missing Values**

In [ ]:
missing_values = data.isnull().sum().sort_values(ascending=False)
missing_percent = (data.isnull().mean() * 100).round(2)

missing_table = pd.concat([missing_values, missing_percent], axis=1)
missing_table.columns = ["missing_count", "missing_percent"]

print(missing_table)

            missing_count  missing_percent
date                    0              0.0
country                 0              0.0
device                  0              0.0
continent               0              0.0
channel                 0              0.0
test                    0              0.0
test_group              0              0.0
event_name              0              0.0
value                   0              0.0


No missing values were found in any column — the dataset is complete and ready for analysis.

###**Tests Structure**

In [ ]:
# Tests

print(f"Unique tests:  {data["test"].nunique()}")
print(f"Test IDs:      {sorted(data["test"].unique().tolist())}")
print(f"Test groups:   {sorted(data["test_group"].unique().tolist())}")

Unique tests:  4
Test IDs:      [1, 2, 3, 4]
Test groups:   [1, 2]


In [ ]:
# Sessions by Test and Group
print("Sessions by Test and Group:")
sessions_overview = (data[data["event_name"] == "session"]
                     .groupby(["test", "test_group"])["value"]
                     .sum()
                     .reset_index()
                     .rename(columns={"value": "sessions"}))
print(sessions_overview.to_string(index=False))

Sessions by Test and Group:
 test  test_group  sessions
    1           1     45362
    1           2     45193
    2           1     50637
    2           2     50244
    3           1     70047
    3           2     70439
    4           1    105079
    4           2    105141


The dataset contains **4 unique A/B tests** (Test 1, 2, 3, 4),
each with **2 groups** — Group 1 (control) and Group 2 (experiment).

Groups are well-balanced across all tests — Group 1 and Group 2
have nearly equal session counts, which is expected in a properly
designed A/B test.

In [ ]:
# Events types
print(f"Unique event types: {data["event_name"].nunique()}")
print("Event names and total values:")
events_overview = (data.groupby("event_name")["value"]
                   .sum()
                   .sort_values(ascending=False)
                   .reset_index())
print(events_overview.to_string(index=False))

Unique event types: 19
Event names and total values:
         event_name   value
          page_view 2150613
    user_engagement 1784891
             scroll  798820
          view_item  653408
      session_start  550685
            session  542142
        first_visit  392482
     view_promotion  309137
        add_to_cart   86626
     begin_checkout   59998
session_with_orders   54324
       new_accounts   45202
        select_item   44417
view_search_results   42208
  add_shipping_info   33815
   add_payment_info   23622
   select_promotion   14978
              click    2627
     view_item_list     133


The dataset contains **19 unique event types**. The 4 key metrics
for statistical analysis are:
- add_payment_info
- add_shipping_info
- begin_checkout
- new_accounts

These metrics represent key steps in the purchase funnel —
from checkout initiation to payment completion and account creation.

##**5. Parameters Setup**

In [ ]:
# Define conversion metrics for analysis
metrics = [
    "add_payment_info",
    "add_shipping_info",
    "begin_checkout",
    "new_accounts"
]

denominator = "session"

alpha = 0.05

##**6. Statistical Functions**

In [ ]:
def run_ztest(sessions_1, events_1, sessions_2, events_2):
    """
    Compare conversion rates between control and test groups using a two-proportion z-test.

    Parameters:
    ----------
    sessions_1, sessions_2 : int
        Number of sessions in control and test groups
    events_1, events_2 : int
        Number of conversion events in control and test groups

    Returns:
    -------
    cr_control : float
        Conversion rate for control group
    cr_test : float
        Conversion rate for test group
    uplift : float
        Relative uplift percentage
    z_stat : float
        Z-statistic value
    p_value : float
        P-value for statistical significance
    """

    # Calculate conversion rates
    cr_control = events_1 / sessions_1 if sessions_1 > 0 else 0
    cr_test = events_2 / sessions_2 if sessions_2 > 0 else 0

    # Calculate uplift percentage
    uplift = ((cr_test - cr_control) / cr_control * 100) if cr_control > 0 else 0

    # Handle cases with zero variance
    if (cr_control == 0 and cr_test == 0) or (cr_control == 1 and cr_test == 1):
        return cr_control, cr_test, uplift, 0.0, 1.0

    # Perform Z-test for two independent proportions
    z_stat, p_value = sm.stats.proportions_ztest(
        [events_1, events_2],
        [sessions_1, sessions_2]
    )

    return cr_control, cr_test, uplift, z_stat, p_value

In [ ]:
def calculate_significance(data, group_cols):
    """
    Calculate conversion rates and statistical significance for multiple metrics.
    Aggregates sessions and events, computes conversion rates, uplift, and applies z-test.

    Parameters:
    ----------
    data : pandas.DataFrame
        Input dataset with event-level data (event_name, value, test_group, etc.)

    group_cols : list of str
        Columns used to group the data (e.g., ['test'], ['test', 'country']).

    Returns:
    -------
    pandas.DataFrame
        DataFrame with conversion metrics, uplift, p-value, and significance flag.
    """

    results = []

    # Get sessions (denominators)
    sessions_data = (
        data[data["event_name"] == denominator]
        .groupby(group_cols + ["test_group"])["value"]
        .sum()
        .rename("sessions")
    )

    # Iterate over all metrics
    for metric in metrics:

        # Get events (nominators)
        events_data = (
            data[data["event_name"] == metric]
            .groupby(group_cols + ["test_group"])["value"]
            .sum()
            .rename("events")
        )

        # Merge sessions and events
        merged = pd.concat([sessions_data, events_data], axis=1).fillna(0).reset_index()

        # Process each group combination
        for keys, group_data in merged.groupby(group_cols):

            groups = group_data.set_index("test_group")

            # Ensure both control and test groups exist
            if not {1, 2}.issubset(groups.index):
                continue

            g1 = groups.loc[1] # Control group
            g2 = groups.loc[2] # Test group

            # Extract values
            sessions_1 = int(g1["sessions"])
            events_1   = int(g1["events"])
            sessions_2 = int(g2["sessions"])
            events_2   = int(g2["events"])

            # Run statistical test
            cr_control, cr_test, uplift, z_stat, p_value = run_ztest(
                sessions_1, events_1,
                sessions_2, events_2
            )

            # Determine significance
            sig_label = "TRUE" if p_value < alpha else "FALSE"

            # Build result row
            group_values = dict(zip(group_cols, keys if isinstance(keys, tuple) else [keys]))

            result_row = {
                **group_values,
                "metric": metric,
                "numerator_event": metric,
                "denominator_event": denominator,
                "numerator_control": events_1,
                "denominator_control": sessions_1,
                "conversion_rate_control": round(cr_control, 6),
                "numerator_test": events_2,
                "denominator_test": sessions_2,
                "conversion_rate_test": round(cr_test, 6),
                "metric_change_pct": round(uplift, 2),
                "z_stat": round(z_stat, 4),
                "p_value": round(p_value, 4),
                "significant": sig_label
            }

            results.append(result_row)

    return pd.DataFrame(results)

##**7. Calculate Statistical Significance**

### **Overall by Test**


In [ ]:
results_total = calculate_significance(data, ["test"])
results_total.head()

,test,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_pct,z_stat,p_value,significant
0,1,add_payment_info,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,12.54,-3.9249,0.0001,TRUE
1,2,add_payment_info,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,3.58,-1.2410,0.2146,FALSE
2,3,add_payment_info,add_payment_info,session,3623,70047,0.051722,3697,70439,0.052485,1.47,-0.6432,0.5201,FALSE
3,4,add_payment_info,add_payment_info,session,3731,105079,0.035507,3601,105141,0.034249,-3.54,1.5711,0.1162,FALSE
4,1,add_shipping_info,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,6.56,-2.6036,0.0092,TRUE


In [ ]:
significant_total = results_total[results_total["significant"] == "TRUE"]

print(f"Significant results: {results_total["significant"].eq("TRUE").sum():,}\n")
print(significant_total[["test", "metric", "conversion_rate_control", "conversion_rate_test",
                         "metric_change_pct", "p_value"]].to_string(index=False))

Significant results: 6

 test            metric  conversion_rate_control  conversion_rate_test  metric_change_pct  p_value
    1  add_payment_info                 0.043825              0.049322              12.54   0.0001
    1 add_shipping_info                 0.066884              0.071272               6.56   0.0092
    1    begin_checkout                 0.083418              0.088974               6.66   0.0029
    3    begin_checkout                 0.136080              0.131518              -3.35   0.0120
    4    begin_checkout                 0.119482              0.116672              -2.35   0.0459
    4      new_accounts                 0.085498              0.082622              -3.36   0.0175


Out of 16 total metric-test combinations, 6 were statistically significant (p < 0.05):

- **Test 1** showed positive uplift across all 3 significant metrics:
  - *add_payment_info* +12.54% (p=0.0001)
  - *add_shipping_info* +6.56% (p=0.0092)
  - *begin_checkout* +6.66% (p=0.0029)
- **Test 2** None of the 4 metrics reached significance. Variant had no measurable impact.
- **Test 3** showed negative uplift for *begin_checkout* -3.35% (p=0.0120)
- **Test 4** showed negative uplift for both:
  - *begin_checkout* -2.35% (p=0.0459)
  - *new_accounts* -3.36% (p=0.0175)

Test 1 is the only test with consistently positive and statistically
significant results across multiple funnel metrics.

### **By Test and Device**


In [ ]:
results_device = calculate_significance(data, ["test","device"])
results_device.head()

,test,device,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_pct,z_stat,p_value,significant
0,1,desktop,add_payment_info,add_payment_info,session,1130,26467,0.042695,1256,26417,0.047545,11.36,-2.6870,0.0072,TRUE
1,1,mobile,add_payment_info,add_payment_info,session,810,17896,0.045262,942,17767,0.053020,17.14,-3.3893,0.0007,TRUE
2,1,tablet,add_payment_info,add_payment_info,session,48,999,0.048048,31,1009,0.030723,-36.06,1.9966,0.0459,TRUE
3,2,desktop,add_payment_info,add_payment_info,session,1314,29497,0.044547,1401,29380,0.047686,7.05,-1.8156,0.0694,FALSE
4,2,mobile,add_payment_info,add_payment_info,session,978,20017,0.048858,961,19756,0.048643,-0.44,0.0996,0.9207,FALSE


In [ ]:
significant_device = results_device[results_device["significant"] == "TRUE"]

print(f"Significant results: {results_device["significant"].eq("TRUE").sum():,}\n")
print(significant_device[["test", "device", "metric", "conversion_rate_control", "conversion_rate_test",
                         "metric_change_pct", "p_value"]].to_string(index=False))

Significant results: 17

 test  device            metric  conversion_rate_control  conversion_rate_test  metric_change_pct  p_value
    1 desktop  add_payment_info                 0.042695              0.047545              11.36   0.0072
    1  mobile  add_payment_info                 0.045262              0.053020              17.14   0.0007
    1  tablet  add_payment_info                 0.048048              0.030723             -36.06   0.0459
    4 desktop  add_payment_info                 0.035961              0.033303              -7.39   0.0108
    4  tablet  add_payment_info                 0.044977              0.022881             -49.13   0.0000
    1 desktop add_shipping_info                 0.064647              0.072529              12.19   0.0003
    4 desktop add_shipping_info                 0.049476              0.046308              -6.40   0.0093
    4  tablet add_shipping_info                 0.052543              0.037288             -29.03   0.0113
    1 deskto

- **Test 1** desktop and mobile show positive uplift for *add_payment_info* and *add_shipping_info*
- **Tablets consistently underperform** across tests 1, 3, and 4 —
  negative uplift for *begin_checkout* and *add_payment_info*. This suggests the test variant may not be optimized for tablet UX

### **By Test and Channel**


In [ ]:
results_channel = calculate_significance(data, ["test","channel"])
results_channel.head()

,test,channel,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_pct,z_stat,p_value,significant
0,1,Direct,add_payment_info,add_payment_info,session,392,10691,0.036666,516,10361,0.049802,35.83,-4.6903,0.0000,TRUE
1,1,Organic Search,add_payment_info,add_payment_info,session,640,15675,0.040829,514,15631,0.032883,-19.46,3.7308,0.0002,TRUE
2,1,Paid Search,add_payment_info,add_payment_info,session,450,11777,0.038210,519,11841,0.043831,14.71,-2.1774,0.0295,TRUE
3,1,Social Search,add_payment_info,add_payment_info,session,237,3883,0.061035,264,3963,0.066616,9.14,-1.0109,0.3121,FALSE
4,1,Undefined,add_payment_info,add_payment_info,session,269,3336,0.080635,416,3397,0.122461,51.87,-5.6762,0.0000,TRUE


In [ ]:
significant_channel = results_channel[results_channel["significant"] == "TRUE"]

print(f"Significant results: {results_channel["significant"].eq("TRUE").sum():,}\n")
print(significant_channel[["test", "channel", "metric", "conversion_rate_control", "conversion_rate_test",
                         "metric_change_pct", "p_value"]].to_string(index=False))

Significant results: 32

 test        channel            metric  conversion_rate_control  conversion_rate_test  metric_change_pct  p_value
    1         Direct  add_payment_info                 0.036666              0.049802              35.83   0.0000
    1 Organic Search  add_payment_info                 0.040829              0.032883             -19.46   0.0002
    1    Paid Search  add_payment_info                 0.038210              0.043831              14.71   0.0295
    1      Undefined  add_payment_info                 0.080635              0.122461              51.87   0.0000
    2 Organic Search  add_payment_info                 0.039963              0.034255             -14.28   0.0048
    2  Social Search  add_payment_info                 0.059390              0.070180              18.17   0.0415
    2      Undefined  add_payment_info                 0.090686              0.118801              31.00   0.0001
    3      Undefined  add_payment_info                 0.106126

- **Undefined channel** consistently shows the strongest positive
  uplift across all tests and metrics — but this channel likely
  represents untracked traffic and results should be interpreted cautiously
- **Organic Search** shows negative uplift in Tests 1, 2, 3, 4 for
  *begin_checkout* — the test variant may be less effective for
  organic traffic
- **Direct channel in Test 1** shows positive uplift for both
  *add_payment_info* +35.83% and *begin_checkout* +14.72%

### **By Test and Continent**


In [ ]:
results_continent = calculate_significance(data, ["test","continent"])
results_continent.head()

,test,continent,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_pct,z_stat,p_value,significant
0,1,(not set),add_payment_info,add_payment_info,session,7,97,0.072165,10,100,0.100000,38.57,-0.6956,0.4867,FALSE
1,1,Africa,add_payment_info,add_payment_info,session,19,494,0.038462,24,474,0.050633,31.65,-0.9188,0.3582,FALSE
2,1,Americas,add_payment_info,add_payment_info,session,1107,25164,0.043991,1177,24769,0.047519,8.02,-1.8865,0.0592,FALSE
3,1,Asia,add_payment_info,add_payment_info,session,512,10626,0.048184,545,10932,0.049854,3.47,-0.5677,0.5702,FALSE
4,1,Europe,add_payment_info,add_payment_info,session,324,8472,0.038244,452,8423,0.053663,40.32,-4.7870,0.0000,TRUE


In [ ]:
significant_continent = results_continent[results_continent["significant"] == "TRUE"]

print(f"Significant results: {results_continent["significant"].eq("TRUE").sum():,}\n")
print(significant_continent[["test", "continent", "metric", "conversion_rate_control", "conversion_rate_test",
                         "metric_change_pct", "p_value"]].to_string(index=False))

Significant results: 28

 test continent            metric  conversion_rate_control  conversion_rate_test  metric_change_pct  p_value
    1    Europe  add_payment_info                 0.038244              0.053663              40.32   0.0000
    2    Africa  add_payment_info                 0.056537              0.022449             -60.29   0.0052
    2  Americas  add_payment_info                 0.044083              0.048322               9.62   0.0172
    2    Europe  add_payment_info                 0.050525              0.042391             -16.10   0.0080
    4 (not set)  add_payment_info                 0.094017              0.030534             -67.52   0.0031
    4    Europe  add_payment_info                 0.037932              0.033490             -11.71   0.0178
    4   Oceania  add_payment_info                 0.068493              0.035547             -48.10   0.0006
    1 (not set) add_shipping_info                 0.041237              0.130000             215.25   0

- **Europe in Test 1** shows strong positive uplift for
  *add_payment_info* +40.32% and *add_shipping_info* +15.44%
- **Africa shows mixed results** — positive in Test 4 for
  *begin_checkout* +43.14% but negative in Tests 2 and 1
- **Oceania shows negative trends** in Tests 2 and 4 for most metrics

### **By Test and Country**


In [ ]:
results_country = calculate_significance(data, ["test","country"])
results_country.head()

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)


,test,country,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_pct,z_stat,p_value,significant
0,1,(not set),add_payment_info,add_payment_info,session,16,369,0.043360,19,373,0.050938,17.48,-0.4868,0.6264,FALSE
1,1,Albania,add_payment_info,add_payment_info,session,1,9,0.111111,0,16,0.000000,-100.00,1.3608,0.1736,FALSE
2,1,Algeria,add_payment_info,add_payment_info,session,2,29,0.068966,3,23,0.130435,89.13,-0.7468,0.4552,FALSE
3,1,Argentina,add_payment_info,add_payment_info,session,6,122,0.049180,2,122,0.016393,-66.67,1.4380,0.1504,FALSE
4,1,Armenia,add_payment_info,add_payment_info,session,0,8,0.000000,0,11,0.000000,0.00,0.0000,1.0000,FALSE


In [ ]:
significant_country = results_country[results_country["significant"] == "TRUE"]

print(f"Significant results: {results_country["significant"].eq("TRUE").sum():,}\n")
print(significant_country[["test", "country", "metric", "conversion_rate_control", "conversion_rate_test",
                         "metric_change_pct", "p_value"]].to_string(index=False))

Significant results: 293

 test              country            metric  conversion_rate_control  conversion_rate_test  metric_change_pct  p_value
    1               Canada  add_payment_info                 0.040380              0.058663              45.28   0.0006
    1           Costa Rica  add_payment_info                 0.312500              0.000000            -100.00   0.0041
    1               France  add_payment_info                 0.038636              0.082393             113.25   0.0001
    1              Georgia  add_payment_info                 0.157895              0.000000            -100.00   0.0360
    1              Hungary  add_payment_info                 0.000000              0.083333               0.00   0.0316
    1              Ireland  add_payment_info                 0.027559              0.092000             233.83   0.0022
    1                Italy  add_payment_info                 0.016000              0.047078             194.24   0.0017
    1         

The country-level analysis reveals the highest granularity with
293 significant results, but many are based on small sample sizes
(countries with very few sessions), which increases the risk of
false positives. Results for major markets (US, UK, Canada, India)
are more reliable.

**Key markets with positive uplift in Test 1:**
- Canada: *begin_checkout* +27.21%, *add_payment_info* +45.28%
- India: *begin_checkout* +26.78%, *add_shipping_info* +23.41%
- Spain: positive across all metrics

### **Key Insights**

1. **Test 1 is the winner** — the only test with statistically
   significant positive uplift across multiple metrics at the
   overall level. Recommend rolling out Test 1 variant to all users.

2. **Tablet experience needs attention** — tablets show negative
   results across multiple tests and metrics. A separate tablet-optimized
   variant should be considered.

3. **Geographic heterogeneity is high** — the same test produces
   opposite results in different countries and continents, suggesting
   that a one-size-fits-all approach may not be optimal.
   Localized variants could improve performance.

4. **Organic Search users respond differently** — negative uplift for
   *begin_checkout* via organic search across multiple tests suggests
   these users have different intent and may need a different funnel approach.

5. **Tests 3 and 4 are harmful** — both show statistically significant
   negative uplift for key metrics. These variants should not be deployed.

## **8. Results Export**

In [ ]:
# Combine all results into single dataframe
all_results = pd.concat([
    results_total.assign(dimension="total"),
    results_country.assign(dimension="country"),
    results_device.assign(dimension="device"),
    results_continent.assign(dimension="continent"),
    results_channel.assign(dimension="channel")
], ignore_index=True)

all_results.head()

,test,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_pct,z_stat,p_value,significant,dimension,country,device,continent,channel
0,1,add_payment_info,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,12.54,-3.9249,0.0001,TRUE,total,NaN,NaN,NaN,NaN
1,2,add_payment_info,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,3.58,-1.2410,0.2146,FALSE,total,NaN,NaN,NaN,NaN
2,3,add_payment_info,add_payment_info,session,3623,70047,0.051722,3697,70439,0.052485,1.47,-0.6432,0.5201,FALSE,total,NaN,NaN,NaN,NaN
3,4,add_payment_info,add_payment_info,session,3731,105079,0.035507,3601,105141,0.034249,-3.54,1.5711,0.1162,FALSE,total,NaN,NaN,NaN,NaN
4,1,add_shipping_info,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,6.56,-2.6036,0.0092,TRUE,total,NaN,NaN,NaN,NaN


In [ ]:
print(f"Total rows: {len(all_results):,}")
print(f"\nSignificant results by dimension:")
print(all_results.groupby("dimension")["significant"]
      .apply(lambda x: (x == "TRUE").sum())
      .reset_index()
      .rename(columns={"significant": "significant_count"})
      .to_string(index=False))

Total rows: 1,968

Significant results by dimension:
dimension  significant_count
  channel                 32
continent                 28
  country                293
   device                 17
    total                  6


In [ ]:
# Export to CSV
all_results.to_csv("ab_test_results.csv")

files.download("ab_test_results.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>